# Building volume — reproducible pipeline

Computes annual **built-up volume** for every Indian district from the
Google Open Buildings 2.5D Temporal dataset. Per-year reductions are exported
to Google Drive (`Export.table.toDrive`), then read back into the notebook.
Async exports sidestep GEE's 5-minute interactive-compute timeout that
`getInfo()` runs into for heavy reductions over hundreds of districts.

This notebook is **pure Python, single kernel** — run it top-to-bottom on
Google Colab. The district boundaries are fetched from GitHub; the rasters
land in *My Drive / India-Built-and-Lit-Buildings*.

**Output:** `bv_annual.csv` in that Drive folder.


## 1 · Mount Drive + Earth Engine

In [ ]:
!pip install -q -U earthengine-api

from google.colab import drive
drive.mount("/content/drive")

import ee
ee.Authenticate()

PROJECT = "gee-ntl-470405"          # <-- your GEE Cloud project id
ee.Initialize(project=PROJECT)
print("Drive mounted, Earth Engine ready.")

## 2 · Parameters

`SCALE_M` is the **resampling knob**. Open Buildings is ~4 m native; reducing
at 100 m is far faster and, summed over a whole district, gives essentially
the same total. Lower it (e.g. 30 m) for a finer, slower run.


In [ ]:
import os
from pathlib import Path

YEARS        = range(2016, 2024)        # Open Buildings 2.5D Temporal: 2016-2023
SCALE_M      = 100                      # reduction scale in metres (resampling knob)
BOUNDARY_URL = "https://raw.githubusercontent.com/xKDR/India-Built-and-Lit/main/data/boundaries/districts_simplified.geojson"
DRIVE_FOLDER = "India-Built-and-Lit-Buildings"
COLLECTION   = "GOOGLE/Research/open-buildings-temporal/v1"
ID_COLUMNS   = ["pc11_s_id", "pc11_d_id", "d_name"]

DRIVE_DIR = Path("/content/drive/MyDrive") / DRIVE_FOLDER

## 3 · Districts → `ee.FeatureCollection`

The simplified SHRUG district polygons are fetched from GitHub and loaded
inline — no GEE asset upload, no local file.


In [ ]:
import json, urllib.request

with urllib.request.urlopen(BOUNDARY_URL) as resp:
    gj = json.load(resp)

feats = [
    ee.Feature(
        ee.Geometry(ft["geometry"], proj="EPSG:4326", geodesic=False),
        {k: ft["properties"].get(k) for k in ID_COLUMNS},
    )
    for ft in gj["features"]
]
districts = ee.FeatureCollection(feats)
print(districts.size().getInfo(), "districts loaded")

## 4 · Queue one Export per year → Drive

`volume = sum(building_height x pixel_area)` per district. The
`building_height` band is 0 on non-building pixels, so the integral over a
district polygon picks up only the built-up area. `footprint_m2` is the area
of pixels carrying any building.

Async `Export.table.toDrive` has no interactive compute timeout — heavy
reductions complete server-side. Already-present per-year CSVs on Drive are
skipped so the cell is resumable.


In [ ]:
def year_image(year):
    img = (ee.ImageCollection(COLLECTION)
           .filterDate(f"{year}-01-01", f"{year + 1}-01-01")
           .mosaic())
    px = ee.Image.pixelArea()
    h  = img.select("building_height")
    volume    = h.multiply(px).rename("volume_m3")
    footprint = px.multiply(h.gt(0)).rename("footprint_m2")
    return volume.addBands(footprint)


have = set(os.listdir(DRIVE_DIR)) if DRIVE_DIR.is_dir() else set()

tasks = {}
for year in YEARS:
    desc = f"buildings_{year}"
    if f"{desc}.csv" in have:
        continue
    fc = (year_image(year)
          .reduceRegions(collection=districts, reducer=ee.Reducer.sum(),
                         scale=SCALE_M, tileScale=16)
          .map(lambda f: f.set("year", year)))
    t = ee.batch.Export.table.toDrive(
        collection=fc, description=desc,
        folder=DRIVE_FOLDER, fileNamePrefix=desc, fileFormat="CSV",
        selectors=ID_COLUMNS + ["year", "volume_m3", "footprint_m2"])
    t.start()
    tasks[desc] = t
print(f"queued {len(tasks)} yearly exports to My Drive / {DRIVE_FOLDER}")

## 5 · Wait for the export tasks

In [ ]:
import time
from tqdm.auto import tqdm

with tqdm(total=len(tasks), desc="exports") as bar:
    done_set = set()
    while True:
        for desc, t in tasks.items():
            if desc in done_set: continue
            if t.status()["state"] in ("COMPLETED", "SUCCEEDED", "FAILED", "CANCELLED"):
                done_set.add(desc)
                bar.update(1)
        if len(done_set) == len(tasks):
            break
        time.sleep(15)
print("All exports finished — CSVs are in", DRIVE_DIR)

## 6 · Assemble → `bv_annual.csv` on Drive

In [ ]:
import pandas as pd

parts = sorted(DRIVE_DIR.glob("buildings_*.csv"))
df = pd.concat((pd.read_csv(p) for p in parts), ignore_index=True)
df = df[df.footprint_m2 > 0].copy()
df["year"] = df["year"].astype(int)
df["mean_height_m"] = df.volume_m3 / df.footprint_m2
df = df[ID_COLUMNS + ["year", "footprint_m2", "volume_m3", "mean_height_m"]]
df = df.sort_values(["pc11_s_id", "pc11_d_id", "year"])

OUT_CSV = DRIVE_DIR / "bv_annual.csv"
df.to_csv(OUT_CSV, index=False)
print(f"wrote {len(df)} rows -> {OUT_CSV}")
df.head()

## 7 · Plot — national built-up volume by year

In [ ]:
import matplotlib.pyplot as plt

national = df.groupby("year").volume_m3.sum()
ax = national.plot(marker="o", color="#f57d6a", figsize=(7, 4))
ax.set_ylabel("Total built-up volume (m^3)")
ax.set_title("India - total built-up volume by year")
ax.grid(alpha=0.3)
plt.show()

## 8 · Plot — choropleth, latest year

In [ ]:
import plotly.express as px

latest = df.year.max()
sub = df[df.year == latest]

fig = px.choropleth(
    sub, geojson=gj, locations="pc11_d_id",
    featureidkey="properties.pc11_d_id",
    color="volume_m3", color_continuous_scale="OrRd",
    hover_name="d_name",
    labels={"volume_m3": "Built-up volume (m³)"},
)
fig.update_geos(fitbounds="locations", visible=False)
fig.update_layout(title=f"Built-up volume by district — {latest}",
                   margin=dict(l=0, r=0, t=40, b=0))
fig.show()

## 9 · Plot — top 20 districts, latest year

In [ ]:
top = sub.nlargest(20, "volume_m3").sort_values("volume_m3")
fig = px.bar(top, x="volume_m3", y="d_name", orientation="h",
             hover_data=["pc11_s_id"],
             labels={"volume_m3": "Built-up volume (m³)", "d_name": "District"},
             color_discrete_sequence=["#f57d6a"])
fig.update_layout(title=f"Top 20 districts by built-up volume — {latest}",
                  height=560, margin=dict(l=10, r=10, t=40, b=10))
fig.show()

## 10 · Annual built-up volume by state (trend)

One line per PC11 state, summed across that state's districts. Mirrors the
"Trend by state" chart on the dashboard.


In [ ]:
PC11_STATES = {
    "1":"Jammu & Kashmir", "2":"Himachal Pradesh", "3":"Punjab",
    "4":"Chandigarh", "5":"Uttarakhand", "6":"Haryana",
    "7":"NCT of Delhi", "8":"Rajasthan", "9":"Uttar Pradesh",
    "10":"Bihar", "11":"Sikkim", "12":"Arunachal Pradesh",
    "13":"Nagaland", "14":"Manipur", "15":"Mizoram",
    "16":"Tripura", "17":"Meghalaya", "18":"Assam",
    "19":"West Bengal", "20":"Jharkhand", "21":"Odisha",
    "22":"Chhattisgarh", "23":"Madhya Pradesh", "24":"Gujarat",
    "25":"Daman & Diu", "26":"Dadra & Nagar Haveli",
    "27":"Maharashtra", "28":"Andhra Pradesh", "29":"Karnataka",
    "30":"Goa", "31":"Lakshadweep", "32":"Kerala",
    "33":"Tamil Nadu", "34":"Puducherry", "35":"Andaman & Nicobar Is.",
}

state_yr = (df.groupby(["pc11_s_id", "year"], as_index=False)
              .volume_m3.sum())
state_yr["state"] = state_yr.pc11_s_id.astype(str).map(PC11_STATES) \
                                                  .fillna(state_yr.pc11_s_id.astype(str))
state_yr = state_yr.sort_values(["state", "year"])

fig = px.line(state_yr, x="year", y="volume_m3", color="state",
              labels={"volume_m3": "Built-up volume (m³)",
                       "year": "Year", "state": "State"})
fig.update_layout(title="Annual built-up volume by state",
                  height=560, margin=dict(l=10, r=10, t=40, b=10))
fig.show()

> **Caveat.** This series is the *raw* Open Buildings 2.5D output — it has
> not been cleaned, and the **2022** snapshot in particular looks problematic.
> Treat it with caution.
>
> `bv_annual.csv` and the per-year inputs are in
> *My Drive / India-Built-and-Lit-Buildings*.
